# Stability Criteria: Input Parameters

Analyses the **input parameters** required by the Roch skier and WEAC skier stability criteria
on ECTP slabs: which calculation pathways are available, how many slabs have all required inputs
(coverage), and the relative uncertainty of each input parameter by method.

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Find Parameterizations](#2-find-parameterizations)
3. [Roch Skier: Input Parameter Analysis](#3-roch-skier-input-parameter-analysis)
4. [WEAC Skier: Input Parameter Analysis](#4-weac-skier-input-parameter-analysis)
5. [Input Coverage Comparison](#5-input-coverage-comparison)
6. [Save Results](#6-save-results)

| Criterion | Target | Required layer inputs | Pathways |
|-----------|--------|-----------------------|----------|
| Roch skier | S_sk = (τ_c − τ) / τ_sk | density | 4 |
| WEAC skier | g_delta = Γ/Γ_c | density + E-mod + ν + G | 32 |

τ_c and weak-layer fracture parameters are constants (weissgraeber_rosendahl) and always available.

Results are saved to parquet for use by `stability_criteria_outputs.ipynb`.

## 1. Setup & Data Loading

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from snowpyt_mechparams.snowpilot import parse_caaml_directory
from snowpyt_mechparams.models import Pit
from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

In [2]:
# ── Helper functions ──────────────────────────────────────────────────────

def nominal(v):
    """Extract nominal float from UFloat or plain float."""
    if v is None:
        return math.nan
    if hasattr(v, 'nominal_value'):
        return float(v.nominal_value)
    try:
        return float(v)
    except (TypeError, ValueError):
        return math.nan


def count_ok(traces, param, n_layers):
    """True if all n_layers have a successful output for param."""
    n = sum(
        1 for t in traces
        if t.parameter == param and t.layer_index is not None
        and t.success and t.output is not None
    )
    return n == n_layers


def rgba(hex_color, alpha=0.75):
    """Convert hex colour to plotly rgba string."""
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'


# ── Wong (2011) colorblind-safe method colours ────────────────────────────
# Consistent with all_density_pathways.ipynb, all_e_mod_pathways.ipynb, etc.
DENSITY_COLORS = {
    'data_flow':           '#CC79A7',   # pink
    'geldsetzer':          '#0072B2',   # blue
    'kim_jamieson_table2': '#009E73',   # green
    'kim_jamieson_table5': '#E69F00',   # orange
}

In [3]:
snow_pits_raw = parse_caaml_directory(str(Path('data')))
pits = [Pit.from_snow_pit(sp) for sp in snow_pits_raw]
print(f'Loaded {len(pits):,} snow pits ({sum(len(p.layers) for p in pits):,} layers)')

ectp_slabs = []
for pit in pits:
    for slab in pit.create_slabs(weak_layer_def='ECTP_failure_layer'):
        ectp_slabs.append(slab)

total_slabs = len(ectp_slabs)
print(f'Created {total_slabs:,} ECTP slabs')

Loaded 50,278 snow pits (371,429 layers)
Created 14,776 ECTP slabs


## 2. Find Parameterizations

In [4]:
engine = ExecutionEngine(graph)

roch_paths = find_parameterizations(graph, graph.get_node('s_sk'))
weac_paths = find_parameterizations(graph, graph.get_node('g_delta'))

print(f'Pathways to s_sk   (Roch skier): {len(roch_paths)}')
print(f'Pathways to g_delta (WEAC skier): {len(weac_paths)}')
print()
print('Roch pathways:')
for i, p in enumerate(roch_paths, 1):
    print(f'  {i}:', p)
    print()

Pathways to s_sk   (Roch skier): 4
Pathways to g_delta (WEAC skier): 32

Roch pathways:
  1: branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_roch_inputs
branch 2: snow_pit -- weissgraeber_rosendahl --> tau_c -- data_flow --> merge_roch_inputs
merge branch 1, branch 2: merge_roch_inputs -- roch_skier --> s_sk

  2: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
branch 3: snow_pit -- weissgraeber_rosendahl --> tau_c -- data_flow --> merge_roch_inputs
merge branch 1, branch 2: merge_hand_hardness_grain_form -- geldsetzer --> density
merge branch 1, branch 2, branch 3: merge_roch_inputs -- roch_skier --> s_sk

  3: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -

## 3. Roch Skier: Input Parameter Analysis

S_sk = (τ_c − τ) / τ_sk requires:
- **density** for every slab layer — 4 methods → 4 pathways
- **τ_c** (constant, weissgraeber_rosendahl — always available)

Coverage is the fraction of ECTP slabs for which **all layers** have a successful density calculation.

In [5]:
config_roch = ExecutionConfig(include_method_uncertainty=False)

roch_slab_records = []   # per (slab, pathway) — for coverage + output

for slab in tqdm(ectp_slabs, desc='Roch s_sk', unit='slab'):
    results  = engine.execute_all(slab, 's_sk', config=config_roch)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        dm = pw.methods_used.get('density', 'data_flow')

        # ── Density traces (slab-level coverage) ───────────────────────────
        d_traces = [
            t for t in pw.computation_trace
            if t.parameter == 'density' and t.layer_index is not None
        ]
        n_ok            = sum(1 for t in d_traces if t.success and t.output is not None)
        slab_density_ok = (n_ok == n_layers)

        # ── s_sk output ────────────────────────────────────────────────────
        roch_res = pw.slab.roch_skier_result
        s_sk_val = np.nan
        if roch_res is not None and roch_res.stability_index is not None:
            s_sk_val = nominal(roch_res.stability_index)
        s_sk_ok = not np.isnan(s_sk_val)

        roch_slab_records.append({
            'slab_id':         slab.slab_id,
            'density_method':  dm,
            'slab_density_ok': slab_density_ok,
            's_sk':            s_sk_val,
            's_sk_ok':         s_sk_ok,
            'unstable_roch':   (s_sk_val < 1.0) if s_sk_ok else None,
        })

roch_slab_df = pd.DataFrame(roch_slab_records)
print(f'Roch slab records:  {len(roch_slab_df):,}')

Roch s_sk: 100%|██████████| 14776/14776 [00:05<00:00, 2838.46slab/s]

Roch slab records:  59,104


In [6]:
roch_cov = (
    roch_slab_df
    .groupby('density_method')
    .agg(
        n_density_ok=('slab_density_ok', 'sum'),
        n_s_sk_ok   =('s_sk_ok',         'sum'),
    )
    .reset_index()
)
roch_cov['pct_density'] = roch_cov['n_density_ok'] / total_slabs
roch_cov['pct_s_sk']    = roch_cov['n_s_sk_ok']    / total_slabs
roch_cov = roch_cov.sort_values('n_s_sk_ok', ascending=False)

print(f"  {'Method':<28}  {'Density OK (all layers)':>28}  {'S_sk OK':>28}")
print(f"  {'-'*90}")
for _, row in roch_cov.iterrows():
    nd = int(row['n_density_ok'])
    ns = int(row['n_s_sk_ok'])
    print(f"  {row['density_method']:<28}  "
          f"{nd:>5} / {total_slabs} ({row['pct_density']:.1%})  "
          f"{ns:>5} / {total_slabs} ({row['pct_s_sk']:.1%})")
print()
print("  'Density OK': all layers in the slab have a successful density calculation.")
print("  'S_sk OK':    Roch skier criterion returned a valid stability index.")

  Method                             Density OK (all layers)                       S_sk OK
  ------------------------------------------------------------------------------------------
  kim_jamieson_table2            5951 / 14776 (40.3%)   5512 / 14776 (37.3%)
  geldsetzer                     4539 / 14776 (30.7%)   4229 / 14776 (28.6%)
  kim_jamieson_table5            1145 / 14776 (7.7%)   1080 / 14776 (7.3%)
  data_flow                       109 / 14776 (0.7%)    109 / 14776 (0.7%)

  'Density OK': all layers in the slab have a successful density calculation.
  'S_sk OK':    Roch skier criterion returned a valid stability index.


## 4. WEAC Skier: Input Parameter Analysis

g_delta = Γ/Γ_c requires four layer-level parameters for **every slab layer**:

| Parameter | Methods | Count |
|-----------|---------|-------|
| density | data_flow, geldsetzer, kim_jamieson_table2, kim_jamieson_table5 | 4 |
| elastic modulus | wautier, kochle, bergfeld, schottner | 4 |
| Poisson's ratio | kochle, srivastava | 2 |
| shear modulus | wautier | 1 |

4 × 4 × 2 × 1 = **32 pathways**. Weak-layer fracture parameters are constants (always available).

**Note:** This cell runs WEAC for every slab with `weac_timeout_seconds=5.0` and collects
per-slab input coverage and g_delta outputs. Expect ~25 minutes runtime.

In [7]:
config_weac = ExecutionConfig(include_method_uncertainty=False, weac_timeout_seconds=5.0)

weac_slab_records = []   # per (slab, pathway) — for coverage + output

for slab in tqdm(ectp_slabs, desc='WEAC g_delta', unit='slab'):
    results  = engine.execute_all(slab, 'g_delta', config=config_weac)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        methods = pw.methods_used
        dm  = methods.get('density',         '?')
        em  = methods.get('elastic_modulus',  '?')
        num = methods.get('poissons_ratio',   '?')

        ok_density = count_ok(pw.computation_trace, 'density',        n_layers)
        ok_emod    = count_ok(pw.computation_trace, 'elastic_modulus', n_layers)
        ok_nu      = count_ok(pw.computation_trace, 'poissons_ratio',  n_layers)
        ok_G       = count_ok(pw.computation_trace, 'shear_modulus',   n_layers)
        all_inputs_ok = ok_density and ok_emod and ok_nu and ok_G

        # ── g_delta output ─────────────────────────────────────────────────
        weac_res = pw.slab.weac_result
        g_val    = np.nan
        if weac_res is not None:
            g_val = float(weac_res.g_delta)
        g_ok = not np.isnan(g_val)

        weac_slab_records.append({
            'slab_id':        slab.slab_id,
            'density_method': dm,
            'emod_method':    em,
            'nu_method':      num,
            'ok_density':     ok_density,
            'ok_emod':        ok_emod,
            'ok_nu':          ok_nu,
            'ok_G':           ok_G,
            'all_inputs_ok':  all_inputs_ok,
            'g_delta':        g_val,
            'g_delta_ok':     g_ok,
            'unstable_weac':  (g_val >= 1.0) if g_ok else None,
        })

weac_slab_df = pd.DataFrame(weac_slab_records)
print(f'WEAC slab records:  {len(weac_slab_df):,}')

WEAC g_delta: 100%|██████████| 14776/14776 [22:19<00:00, 11.03slab/s]  


WEAC slab records:  472,832


In [8]:
weac_cov = (
    weac_slab_df
    .groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        n_density_ok  =('ok_density',     'sum'),
        n_emod_ok     =('ok_emod',         'sum'),
        n_nu_ok       =('ok_nu',           'sum'),
        n_G_ok        =('ok_G',            'sum'),
        n_all_inputs  =('all_inputs_ok',   'sum'),
        n_g_delta_ok  =('g_delta_ok',      'sum'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

def fmt(n):
    return f'{int(n):>4} ({int(n) / total_slabs:.1%})'

print(f"  {'Pathway':<55}  {'ρ':>14}  {'E':>14}  {'ν':>14}  {'G':>14}  {'All inputs':>14}  {'g_delta':>14}")
print(f"  {'-'*145}")
for _, row in weac_cov.iterrows():
    pw = f"{row['density_method']} → {row['emod_method']} → {row['nu_method']}"
    print(f"  {pw:<55}  {fmt(row['n_density_ok']):>14}  {fmt(row['n_emod_ok']):>14}  "
          f"{fmt(row['n_nu_ok']):>14}  {fmt(row['n_G_ok']):>14}  "
          f"{fmt(row['n_all_inputs']):>14}  {fmt(row['n_g_delta_ok']):>14}")
print()
print("  Coverage = % ECTP slabs where all layers have the parameter computed.")

  Pathway                                                               ρ               E               ν               G      All inputs         g_delta
  -------------------------------------------------------------------------------------------------------------------------------------------------
  kim_jamieson_table2 → wautier → kochle                     5951 (40.3%)    2092 (14.2%)      926 (6.3%)    2092 (14.2%)      737 (5.0%)        6 (0.0%)
  kim_jamieson_table2 → schottner → kochle                   5951 (40.3%)    2780 (18.8%)      926 (6.3%)    2092 (14.2%)      737 (5.0%)        9 (0.1%)
  geldsetzer → wautier → kochle                              4539 (30.7%)    1607 (10.9%)      926 (6.3%)    1607 (10.9%)      737 (5.0%)        6 (0.0%)
  geldsetzer → schottner → kochle                            4539 (30.7%)    2780 (18.8%)      926 (6.3%)    1607 (10.9%)      737 (5.0%)        9 (0.1%)
  geldsetzer → kochle → kochle                               4539 (30.7%)      739

In [9]:
TOP_N = 12
top_pw = weac_cov.head(TOP_N).copy()
top_pw['label'] = (
    top_pw['density_method'] + ' → ' +
    top_pw['emod_method']    + ' → ' +
    top_pw['nu_method']
)

steps        = ['n_density_ok', 'n_emod_ok', 'n_nu_ok', 'n_G_ok', 'n_all_inputs', 'n_g_delta_ok']
step_labels  = ['ρ (density)', 'E (elastic mod.)', 'ν (Poisson)', 'G (shear mod.)', 'All inputs', 'g_delta']
step_colors  = ['#0072B2',     '#009E73',           '#E69F00',     '#CC79A7',        '#56B4E9',    '#D55E00']

fig = go.Figure()
for step, label, color in zip(steps, step_labels, step_colors):
    vals = (top_pw[step] / total_slabs * 100).tolist()
    fig.add_trace(go.Bar(
        name=label,
        x=top_pw['label'].tolist(),
        y=vals,
        marker_color=rgba(color, 0.80),
        text=[f'{v:.1f}%' for v in vals],
        textposition='outside',
        textfont=dict(size=8),
    ))

fig.update_layout(
    title=dict(
        text=(
            f'<b>WEAC Skier — Input Parameter Coverage (Top {TOP_N} Pathways by All-Input Coverage)</b><br>'
            '<sup>Coverage = % ECTP slabs where all layers have the parameter computed</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    barmode='group',
    xaxis=dict(tickangle=-35, tickfont=dict(size=9)),
    yaxis=dict(title='Coverage (%)', gridcolor='rgba(200,200,200,0.4)'),
    plot_bgcolor='white',
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.30),
    width=1100, height=520,
    margin=dict(l=60, r=40, t=90, b=180),
)
fig.show()

## 5. Input Coverage Comparison

Roch skier requires only density; WEAC requires density + E-mod + ν + G.
The left panel shows Roch coverage per density method; the right panel shows WEAC coverage for the top
pathways. Colour encodes the density method, consistent across both panels.

In [10]:
TOP_WEAC = 12

roch_sorted = roch_cov.sort_values('pct_s_sk')
weac_top    = weac_cov.head(TOP_WEAC).copy()
weac_top['label'] = (
    weac_top['density_method'] + ' → ' +
    weac_top['emod_method']    + ' → ' +
    weac_top['nu_method']
)
weac_top = weac_top.sort_values('n_all_inputs')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Roch Skier (density input only, 4 pathways)',
        f'WEAC Skier (all 4 layer inputs, top {TOP_WEAC} pathways)',
    ],
    horizontal_spacing=0.14,
)

for _, row in roch_sorted.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['pct_s_sk'] * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[dm], orientation='h',
            marker_color=rgba(color, 0.80),
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=1,
    )

for _, row in weac_top.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['n_all_inputs'] / total_slabs * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[row['label']], orientation='h',
            marker_color=rgba(color, 0.55),
            marker_line_color=rgba(color, 0.90),
            marker_line_width=1.5,
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=2,
    )

fig.update_xaxes(title_text='Coverage (%)', gridcolor='rgba(200,200,200,0.4)')
fig.update_yaxes(tickfont=dict(size=9))
fig.update_layout(
    title=dict(
        text=(
            '<b>Input Coverage: Roch Skier vs WEAC Skier</b><br>'
            '<sup>% ECTP slabs with all required layer inputs computed — colour = density method</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    plot_bgcolor='white',
    width=1100, height=480,
    margin=dict(l=60, r=80, t=90, b=60),
)
fig.show()

best_roch = roch_cov['pct_s_sk'].max() * 100
best_weac = weac_cov['n_all_inputs'].max() / total_slabs * 100
print(f'Best Roch pathway coverage:  {best_roch:.1f}%')
print(f'Best WEAC pathway coverage:  {best_weac:.1f}%')
print(f'Coverage gap (best Roch − best WEAC): {best_roch - best_weac:.1f} percentage points')

Best Roch pathway coverage:  37.3%
Best WEAC pathway coverage:  5.0%
Coverage gap (best Roch − best WEAC): 32.3 percentage points


## 6. Save Results

Saves `roch_slab_df` and `weac_slab_df` to parquet files for use by
`stability_criteria_outputs.ipynb`, which loads these directly without re-running
the ~25-minute WEAC execution.

In [14]:
roch_slab_df.to_parquet('roch_slab_results.parquet', index=False, engine='fastparquet')
weac_slab_df.to_parquet('weac_slab_results.parquet', index=False, engine='fastparquet')
Path('total_slabs.txt').write_text(str(total_slabs))

print(f'Saved roch_slab_results.parquet  ({len(roch_slab_df):,} rows)')
print(f'Saved weac_slab_results.parquet  ({len(weac_slab_df):,} rows)')
print(f'Saved total_slabs.txt            ({total_slabs:,})')
print()
print('Run stability_criteria_outputs.ipynb to compare stability criterion outputs.')

Saved roch_slab_results.parquet  (59,104 rows)
Saved weac_slab_results.parquet  (472,832 rows)
Saved total_slabs.txt            (14,776)

Run stability_criteria_outputs.ipynb to compare stability criterion outputs.
